# 02 - Análisis Descriptivo y Pruebas de Normalidad

El objetivo de este notebook es describir la muestra del estudio de osteoporosis y probar los supuestos de distribución de las variables continuas.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import shapiro

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [2]:
PROCESSED_DATA_DIR = Path(
    "/home/marcos-maravilla/análisis_estadístico_osteoporosis/data/processed"
)
PARQUET_PATH = PROCESSED_DATA_DIR / "BD_Clean_Osteoporosis.parquet"
PICKLE_PATH = PROCESSED_DATA_DIR / "BD_Clean_Osteoporosis.pkl"

if PARQUET_PATH.exists():
    try:
        df = pd.read_parquet(PARQUET_PATH)
        loaded_path = PARQUET_PATH
    except Exception as error:
        print("No se pudo cargar el archivo Parquet; se intentará cargar el archivo Pickle.")
        print(f"Detalle del error: {type(error).__name__}: {error}")
        if not PICKLE_PATH.exists():
            raise FileNotFoundError(f"No se encontró archivo alternativo: {PICKLE_PATH}") from error
        df = pd.read_pickle(PICKLE_PATH)
        loaded_path = PICKLE_PATH
elif PICKLE_PATH.exists():
    df = pd.read_pickle(PICKLE_PATH)
    loaded_path = PICKLE_PATH
else:
    raise FileNotFoundError(
        f"No se encontró ningún archivo procesado en {PROCESSED_DATA_DIR}"
    )

df_clean = df.copy()

print(f"Dataset cargado desde: {loaded_path}")
print(f"Dimensiones del dataset: {df_clean.shape[0]:,} filas y {df_clean.shape[1]:,} columnas")

Dataset cargado desde: /home/marcos-maravilla/análisis_estadístico_osteoporosis/data/processed/BD_Clean_Osteoporosis.parquet
Dimensiones del dataset: 405 filas y 75 columnas


## 1. Pruebas de Normalidad (Shapiro-Wilk)

In [3]:
continuous_variables = ["edad", "peso_(kg)", "altura_(cm)", "imc"]

for variable in continuous_variables:
    values = pd.to_numeric(df[variable], errors="coerce").dropna()
    statistic, p_value = shapiro(values)
    interpretation = "Distribución No Normal" if p_value < 0.05 else "Distribución Normal"

    print(f"{variable}")
    print(f"  W = {statistic:.4f}")
    print(f"  p-valor = {p_value:.4g}")
    print(f"  Interpretación: {interpretation}\n")

edad
  W = 0.9770
  p-valor = 4.869e-06
  Interpretación: Distribución No Normal

peso_(kg)
  W = 0.9850
  p-valor = 0.0003378
  Interpretación: Distribución No Normal

altura_(cm)
  W = 0.9714
  p-valor = 3.933e-07
  Interpretación: Distribución No Normal

imc
  W = 0.9792
  p-valor = 1.429e-05
  Interpretación: Distribución No Normal



## 2. Estadística Descriptiva

In [4]:
descriptive_continuous = []

for variable in continuous_variables:
    values = pd.to_numeric(df[variable], errors="coerce").dropna()
    p25 = values.quantile(0.25)
    median = values.median()
    p75 = values.quantile(0.75)

    descriptive_continuous.append(
        {
            "variable": variable,
            "mediana": median,
            "p25": p25,
            "p75": p75,
            "rango_intercuartilico": f"{p25:.2f} - {p75:.2f}",
        }
    )

continuous_summary = pd.DataFrame(descriptive_continuous)
continuous_summary

,variable,mediana,p25,p75,rango_intercuartilico
0,edad,65.000000,59.00000,72.00000,59.00 - 72.00
1,peso_(kg),69.000000,62.00000,77.00000,62.00 - 77.00
2,altura_(cm),155.000000,151.00000,162.00000,151.00 - 162.00
3,imc,28.194284,25.28257,31.23141,25.28 - 31.23


In [5]:
categorical_variables = ["sexo", "alteracion_osea", "trabaja", "realiza_af"]

for variable in categorical_variables:
    frequency_table = (
        df[variable]
        .value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="frecuencia_absoluta")
    )
    frequency_table["porcentaje"] = (
        frequency_table["frecuencia_absoluta"] / len(df) * 100
    ).round(2)

    print(f"\nVariable: {variable}")
    display(frequency_table)


Variable: sexo


,sexo,frecuencia_absoluta,porcentaje
0,1,327,80.74
1,0,78,19.26



Variable: alteracion_osea


,alteracion_osea,frecuencia_absoluta,porcentaje
0,1,344,84.94
1,0,61,15.06



Variable: trabaja


,trabaja,frecuencia_absoluta,porcentaje
0,0,308,76.05
1,1,97,23.95



Variable: realiza_af


,realiza_af,frecuencia_absoluta,porcentaje
0,1,238,58.77
1,0,167,41.23


## 3. Tabla 1: Características Generales (Variables Continuas)

Se reportará Mediana (RIC 25-75), valor p mediante U de Mann-Whitney y tamaño de efecto mediante V de Cliff.

In [ ]:
from scipy.stats import mannwhitneyu

analysis_df = df_clean.copy() if "df_clean" in globals() else df.copy()

def cliffs_delta(group_0, group_1):
    """Calcula V de Cliff; valores positivos indican mayores valores en el grupo 1."""
    group_0 = np.asarray(group_0, dtype=float)
    group_1 = np.asarray(group_1, dtype=float)

    group_0 = group_0[~np.isnan(group_0)]
    group_1 = group_1[~np.isnan(group_1)]

    if group_0.size == 0 or group_1.size == 0:
        return np.nan

    pairwise_differences = group_1[:, None] - group_0[None, :]
    greater = np.sum(pairwise_differences > 0)
    lower = np.sum(pairwise_differences < 0)
    return (greater - lower) / pairwise_differences.size

def median_iqr(values):
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return "NA"

    median = values.median()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    return f"{median:.2f} ({q1:.2f}, {q3:.2f})"

continuous_variables = ["edad", "imc", "peso_(kg)", "altura_(cm)"]
outcome = pd.to_numeric(analysis_df["alteracion_osea"].astype("string"), errors="coerce")

tabla_1_rows = []

for variable in continuous_variables:
    values = pd.to_numeric(analysis_df[variable], errors="coerce")
    group_0 = values[outcome.eq(0)].dropna()
    group_1 = values[outcome.eq(1)].dropna()

    _, p_value = mannwhitneyu(group_0, group_1, alternative="two-sided")
    effect_size = cliffs_delta(group_0, group_1)

    tabla_1_rows.append(
        {
            "Variable": variable,
            "All participants": median_iqr(values),
            "Salud ósea normal": median_iqr(group_0),
            "Salud ósea alterada": median_iqr(group_1),
            "p-value": f"{p_value:.4f}",
            "Effect size": f"{effect_size:.3f}",
        }
    )

tabla_1 = pd.DataFrame(tabla_1_rows)
display(tabla_1)


## 4. Tabla 2: Características Sociodemográficas y Estilos de Vida (Categóricas)

Se reportarán frecuencias, porcentajes por fila y valor p mediante Chi-cuadrada o prueba exacta de Fisher cuando corresponda.

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact

categorical_variables = {
    "sexo": "Sexo",
    "estado_civil": "Estado civil",
    "lugar_residencia_municipio": "Lugar de residencia",
    "escolaridad": "Escolaridad",
    "trabaja": "Situación laboral",
    "realiza_af": "Actividad física",
    "imc_cat": "Categoría de IMC",
}

outcome = pd.to_numeric(analysis_df["alteracion_osea"].astype("string"), errors="coerce")
tabla_2_rows = []

for variable, variable_label in categorical_variables.items():
    categories = analysis_df[variable].astype("string").fillna("Sin dato")
    contingency_table = pd.crosstab(categories, outcome)

    for group_value in [0, 1]:
        if group_value not in contingency_table.columns:
            contingency_table[group_value] = 0

    contingency_table = contingency_table[[0, 1]].sort_index()
    chi2, chi2_p_value, _, expected = chi2_contingency(contingency_table)
    expected_has_low_count = (expected < 5).any()

    if expected_has_low_count and contingency_table.shape == (2, 2):
        _, p_value = fisher_exact(contingency_table.to_numpy())
        test_used = "Fisher"
    else:
        p_value = chi2_p_value
        test_used = "Chi-cuadrada"

    variable_total = contingency_table.to_numpy().sum()

    for row_index, (category, counts) in enumerate(contingency_table.iterrows()):
        normal_count = int(counts[0])
        altered_count = int(counts[1])
        row_total = normal_count + altered_count
        total_percent = row_total / variable_total * 100 if variable_total else np.nan
        normal_row_percent = normal_count / row_total * 100 if row_total else np.nan
        altered_row_percent = altered_count / row_total * 100 if row_total else np.nan

        tabla_2_rows.append(
            {
                "Variable": variable_label if row_index == 0 else "",
                "Category": category,
                "Total N (%)": f"{row_total} ({total_percent:.1f})",
                "Salud ósea normal n (% row)": f"{normal_count} ({normal_row_percent:.1f})",
                "Salud ósea alterada n (% row)": f"{altered_count} ({altered_row_percent:.1f})",
                "p-value": f"{p_value:.4f}" if row_index == 0 else "",
                "Test": test_used if row_index == 0 else "",
            }
        )

tabla_2 = pd.DataFrame(tabla_2_rows)
display(tabla_2)
